In [1]:
# Part 1: Imports
# ipynb

# 1. Standard library imports
import argparse
import math
import os
from pprint import pprint
import random
import shutil
import time

# 2. Third-party library imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 3. PyTorch core and utilities
import torch
from torch import autograd
from torch.distributions.multivariate_normal import MultivariateNormal
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
!pip install tensorboard
from torch.utils.tensorboard import SummaryWriter

# 4. Torchvision imports
from torchvision import datasets, transforms
from torchvision.utils import save_image

# 5. Device configuration
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else "cpu")


  Using cached tensorboard-2.21.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metadata (1.1 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.0 kB)
Using cached tensorboard-2.21.0-py3-none-any.whl (5.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 22.0 MB/s eta 0:00:00


In [2]:
# ============================================================================
# Part 2: Core Probability, Loss, & Distribution Utilities (Fixed)
# ============================================================================

# Global BCE loss instance (reduction='none' to allow sample-wise reductions)
bce = nn.BCEWithLogitsLoss(reduction='none')


def sample_gaussian(m: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Reparameterization trick for independent diagonal Gaussians: z = mu + eps * std
    m: (batch, ...) Mean
    v: (batch, ...) Variance
    """
    v_clamped = torch.clamp(v, min=1e-7, max=1e7)
    std = torch.sqrt(v_clamped)
    eps = torch.randn_like(std)
    return m + eps * std


def condition_prior(scale: torch.Tensor, label: torch.Tensor, dim: int):
    """
    Vectorized condition prior scaling across batches: N((u - min) / range, I)
    scale: (num_labels, 2) where scale[j] = [min_val, range_val]
    label: (batch, num_labels)
    """
    scale_min = scale[:, 0].unsqueeze(0)    # (1, num_labels)
    scale_range = scale[:, 1].unsqueeze(0)  # (1, num_labels)

    # Normalized multipliers
    mul = ((label - scale_min) / (scale_range + 1e-8)).unsqueeze(-1)  # (batch, num_labels, 1)

    mean = mul.repeat(1, 1, dim)
    var = torch.ones_like(mean)
    return mean, var


def gaussian_parameters(h: torch.Tensor, dim: int = -1):
    """
    Splits tensor along `dim` into mean and strictly positive variance via softplus.
    """
    m, h_var = torch.split(h, h.size(dim) // 2, dim=dim)
    v = F.softplus(h_var) + 1e-7
    return m, v


def log_bernoulli_with_logits(x: torch.Tensor, logits: torch.Tensor) -> torch.Tensor:
    """
    Numerically stable Bernoulli log-likelihood per sample.
    Averages over spatial/channel dimensions to keep loss scales balanced with KL and DAG penalties.
    """
    # Safety clamp targets to valid probability range [0, 1]
    x_clamped = torch.clamp(x, min=0.0, max=1.0)
    loss = -bce(logits, x_clamped.to(logits.dtype))

    # Average across channel and spatial dimensions (dim 1, 2, 3) per sample in batch
    dims = list(range(1, loss.ndim))
    return loss.mean(dim=dims) if dims else loss


def kl_normal(qm: torch.Tensor, qv: torch.Tensor, pm: torch.Tensor, pv: torch.Tensor) -> torch.Tensor:
    """
    Batched analytical KL divergence between two diagonal Gaussians: KL(q || p).
    Sums across latent feature dimensions per sample.
    """
    qv = torch.clamp(qv, min=1e-7, max=1e7)
    pv = torch.clamp(pv, min=1e-7, max=1e7)
    element_wise = 0.5 * (torch.log(pv) - torch.log(qv) + (qv / pv) + ((qm - pm) ** 2) / pv - 1.0)

    dims = list(range(1, element_wise.ndim))
    return element_wise.sum(dim=dims) if dims else element_wise


def load_model_by_name(model: torch.nn.Module, checkpoint_dir: str = 'checkpoints'):
    """Safely loads model weights from disk to the active device."""
    model_name = getattr(model, 'name', 'model')
    file_path = os.path.join(checkpoint_dir, model_name, 'model.pt')

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"No checkpoint found at: {file_path}")

    state = torch.load(file_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    print(f"Successfully loaded weights from {file_path}")


In [3]:
# ============================================================================
# Part 3: Causal DAG Constraints, Network Init, & Dataset Loader
# ============================================================================

def prune(A: torch.Tensor, threshold: float = 0.3) -> torch.Tensor:
    """Zeroes out adjacency weights below a threshold magnitude."""
    return torch.where(torch.abs(A) < threshold, torch.zeros_like(A), A)


def filldiag_zero(A: torch.Tensor) -> torch.Tensor:
    """Sets the diagonal elements of a square matrix to zero in-place."""
    mask = torch.eye(A.size(0), dtype=torch.bool, device=A.device)
    A.masked_fill_(mask, 0.0)
    return A


def _h_A(A: torch.Tensor, m: int) -> torch.Tensor:
    """
    NOTEARS characterization of DAG acyclicity constraint: h(A) = tr(e^(A * A)) - m = 0.
    Using native torch.matrix_exp for numerical precision and gradient stability.
    """
    A_sq = A * A
    expm_A = torch.matrix_exp(A_sq)
    return torch.trace(expm_A) - float(m)


def weights_init(m: nn.Module):
    """Initializes weights for Conv, Linear, and BatchNorm layers."""
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight.data, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)


class dataload_withlabel(Dataset):
    """
    Loads images and extracts ground truth causal factors encoded in filenames
    (e.g., 'a_5_10_34_6.png' -> labels [5.0, 10.0, 34.0, 6.0]).
    """
    def __init__(self, root: str, dataset: str = "train", transform=None):
        super().__init__()
        target_dir = os.path.join(root, dataset)
        valid_exts = {'.png', '.jpg', '.jpeg', '.bmp'}

        filenames = [f for f in sorted(os.listdir(target_dir)) if os.path.splitext(f)[-1].lower() in valid_exts]
        self.imgs = [os.path.join(target_dir, f) for f in filenames]

        self.imglabel = [
            [float(val) for val in os.path.splitext(f)[0].split('_')[1:]]
            for f in filenames
        ]
        self.transforms = transform or transforms.Compose([
            transforms.Resize((96, 96)),  # <--- Add this line
            transforms.ToTensor()
        ])

    def __getitem__(self, idx: int):
        img_path = self.imgs[idx]
        label = torch.tensor(self.imglabel[idx], dtype=torch.float32)

        with Image.open(img_path) as img:
            pil_img = img.convert('RGB')

        data = self.transforms(pil_img)
        return data, label

    def __len__(self):
        return len(self.imgs)


def get_batch_unin_dataset_withlabel(dataset_dir: str, batch_size: int, dataset: str = "train", num_workers: int = 2):
    """Creates a DataLoader for the labeled causal dataset."""
    ds = dataload_withlabel(dataset_dir, dataset=dataset)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=(dataset == "train"),
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )


def get_parse_args():
    """Command-line argument parser."""
    parser = argparse.ArgumentParser(description="Causal DAG VAE Configuration")
    parser.add_argument('--data_dir', type=str, default='./causal_data/flow_noise', help='Path to dataset root')
    parser.add_argument('--batch_size', type=int, default=64, help='Training mini-batch size')
    parser.add_argument('--epochs', type=int, default=100, help='Total training epochs')
    parser.add_argument('--lr', type=float, default=1e-3, help='Learning rate')
    return parser.parse_args()


In [4]:
# ============================================================================
# Part 4: Synthetic Causal Water-Flow Dataset Generation
# ============================================================================

import matplotlib
matplotlib.use('Agg')  # Fast headless backend
import matplotlib.pyplot as plt


def generate_flow_dataset(
    output_dir: str = './causal_data/flow_noise',
    test_split_ratio: int = 5,  # 1 in every 5 images goes to test (20%)
    seed: int = 42
):
    """
    Generates synthetic causal images of a cup leaking fluid with causal dependencies:
    - ball_r -> water_height (h)
    - h, deep -> water_trajectory (x_true)
    """
    np.random.seed(seed)

    train_dir = os.path.join(output_dir, 'train')
    test_dir = os.path.join(output_dir, 'test')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    print(f"Generating synthetic causal dataset at '{output_dir}'...")

    total_samples = 30 * 30 * 9  # 8,100 images
    count = 0
    generated_count = 0
    start_time = time.time()

    plt.ioff()

    for r in range(5, 35):
        ball_r = r / 30.0
        for h_raw in range(10, 40):
            h = pow(ball_r, 3) + (h_raw / 10.0)
            for hole in range(6, 15):
                deep = hole / 3.0

                fig, ax = plt.subplots(figsize=(1.0, 1.0), dpi=96)

                # 1. Water in cup
                rect = plt.Rectangle((3.0, 0.0), 5.0, 5.0 + h, color='lightskyblue')
                ax.add_patch(rect)

                # 2. Submerged ball
                ball = plt.Circle((5.5, ball_r + 0.5), ball_r, color='firebrick', zorder=3)
                ax.add_patch(ball)

                # 3. Cup boundaries (polygons)
                left = plt.Polygon([[3.0, 0.0], [3.0, 19.0]], color='black', linewidth=2)
                right_1 = plt.Polygon([[8.0, 0.0], [8.0, deep]], color='black', linewidth=2)
                right_2 = plt.Polygon([[8.0, deep + 0.4], [8.0, 19.0]], color='black', linewidth=2)
                ax.add_patch(left)
                ax.add_patch(right_1)
                ax.add_patch(right_2)

                # 4. Parabolic water trajectory with noise
                y = np.linspace(deep, 0.5, num=50)
                epsilon = 0.01 * np.max([np.abs(np.random.randn()), 1.0])
                x = np.sqrt(2.0 * (0.98 + epsilon) * h * np.maximum(deep - y, 0.0)) + 8.0
                x_true = np.sqrt(2.0 * 0.98 * h * np.maximum(deep - 0.5, 0.0))

                ax.plot(x, y, color='lightskyblue', linewidth=2)

                # 5. Ground level
                x_ground = np.linspace(0.0, 20.0, num=50)
                y_ground = np.zeros(50) + 0.2
                ax.plot(x_ground, y_ground, color='black', linewidth=2)

                # Frame settings
                ax.set_xlim(0, 20)
                ax.set_ylim(0, 20)
                ax.axis('off')

                # Filename encodes full precision causal parameters to prevent overwriting
                # labels extracted: [r, h, x_true, hole]
                filename = f"a_{float(r):.2f}_{float(h):.2f}_{float(x_true):.2f}_{float(hole):.2f}.png"

                if count % test_split_ratio == (test_split_ratio - 1):
                    save_path = os.path.join(test_dir, filename)
                else:
                    save_path = os.path.join(train_dir, filename)

                fig.savefig(save_path, dpi=96, bbox_inches='tight', pad_inches=0)
                plt.close(fig)

                count += 1
                generated_count += 1

                if generated_count % 1000 == 0:
                    elapsed = time.time() - start_time
                    print(f"Rendered {generated_count}/{total_samples} images ({elapsed:.1f}s elapsed)")

    print(f"Finished generating {generated_count} causal samples in {time.time() - start_time:.2f}s.")


In [5]:
# ============================================================================
# Part 5: CausalVAE Model Architecture (Fixed)
# ============================================================================

class CausalVAE(nn.Module):
    """
    CausalVAE: Disentangled Representation Learning via Neural Structural Causal Models.
    Coordinates Encoder, DAG (Adjacency Matrix A), Masking/SEM layers, and Decoder.
    """
    def __init__(
        self,
        encoder: nn.Module,
        decoder: nn.Module,
        dag_layer: nn.Module,
        attention_layer: nn.Module,
        mask_z_layer: nn.Module,
        mask_u_layer: nn.Module,
        name: str = 'causal_vae',
        z_dim: int = 16,
        z1_dim: int = 4,
        z2_dim: int = 4,
        scale: torch.Tensor = None
    ):
        super().__init__()
        self.name = name
        self.z_dim = z_dim    # Total latent dimension (z1_dim * z2_dim)
        self.z1_dim = z1_dim  # Number of causal concept variables (DAG nodes)
        self.z2_dim = z2_dim  # Latent representation dimension per concept

        # Dataset scale priors: [min_val, range_val] for each causal factor
        if scale is None:
            scale = torch.tensor([
                [5.0, 30.0],    # ball_r
                [1.0, 4.0],     # h (water height)
                [0.0, 30.0],    # x_true (trajectory distance)
                [2.0, 3.0]      # hole / deep
            ], dtype=torch.float32)
        self.register_buffer('scale', scale)

        # Neural Submodules
        self.enc = encoder
        self.dec = decoder
        self.dag = dag_layer
        self.attn = attention_layer
        self.mask_z = mask_z_layer
        self.mask_u = mask_u_layer

        self.mse_loss = nn.MSELoss()

    def negative_elbo_bound(
        self,
        x: torch.Tensor,
        label: torch.Tensor,
        mask: int = None,
        adj: torch.Tensor = None,
        lambdav: float = 0.001,
        beta: float = 1.0
    ):
        """
        Computes Negative Evidence Lower Bound (NELBO), KL divergence, and reconstruction loss.
        Supports interventions via (mask, adj).
        """
        batch_size = x.size(0)
        dev = x.device
        assert label.size(1) == self.z1_dim, f"Label dim ({label.size(1)}) must match z1_dim ({self.z1_dim})"

        # 1. Variational Posterior Encoding: q(z | x)
        q_m, q_v = self.enc.encode(x)
        q_m = q_m.reshape(batch_size, self.z1_dim, self.z2_dim)
        q_v = q_v.reshape(batch_size, self.z1_dim, self.z2_dim)

        # 2. Structural Equation Model over DAG: z = (I - A^T)^(-1) epsilon
        decode_m, decode_v = self.dag.calculate_dag(q_m, q_v)
        decode_m = decode_m.reshape(batch_size, self.z1_dim, self.z2_dim)

        # 3. Interventions / Masking for Counterfactuals
        if mask is not None and mask in [0, 1, 3] and adj is not None:
            z_mask = torch.ones(batch_size, self.z1_dim, self.z2_dim, device=dev) * float(adj)
            decode_m[:, mask, :] = z_mask[:, mask, :]
            decode_v[:, mask, :] = z_mask[:, mask, :]

        m_zm = self.dag.mask_z(decode_m).reshape(batch_size, self.z1_dim, self.z2_dim)
        m_u = self.dag.mask_u(label)

        # 4. Non-linear mixing and Attention mechanism
        f_z = self.mask_z.mix(m_zm).reshape(batch_size, self.z1_dim, self.z2_dim)
        attn_out = self.attn.attention(decode_m, q_m)
        e_tilde = attn_out[0] if isinstance(attn_out, (tuple, list)) else attn_out

        f_z1 = f_z + e_tilde
        if mask is not None and mask == 2 and adj is not None:
            z_mask = torch.ones(batch_size, self.z1_dim, self.z2_dim, device=dev) * float(adj)
            f_z1[:, mask, :] = z_mask[:, mask, :]
            decode_v[:, mask, :] = z_mask[:, mask, :]

        g_u = self.mask_u.mix(m_u)

        # 5. Reparameterized Sampling of Causal Latents
        z_given_dag = sample_gaussian(f_z1, q_v * lambdav)
        z_flat = z_given_dag.reshape(batch_size, self.z_dim)

        # 6. Decoder Reconstruction
        decoded_outputs = self.dec.decode_sep(z_flat, label)
        decoded_logits = decoded_outputs[0] if isinstance(decoded_outputs, (tuple, list)) else decoded_outputs
        decoded_logits = decoded_logits.reshape(x.shape)

        rec = log_bernoulli_with_logits(x, decoded_logits)
        rec_loss = -torch.mean(rec)

        # 7. Priors & KL Divergences
        p_m = torch.zeros(batch_size, self.z_dim, device=dev)
        p_v = torch.ones(batch_size, self.z_dim, device=dev)
        cp_m, cp_v = condition_prior(self.scale, label, self.z2_dim)

        # Base encoder regularizing KL
        kl_base = kl_normal(
            q_m.reshape(batch_size, self.z_dim),
            q_v.reshape(batch_size, self.z_dim),
            p_m,
            p_v
        )

        # Causal Alignment KL across DAG concept nodes
        kl_dag = torch.zeros(1, device=dev)
        mask_kl = torch.zeros(1, device=dev)
        for i in range(self.z1_dim):
            # Prior condition: cp_m, cp_v
            kl_dag = kl_dag + torch.mean(kl_normal(decode_m[:, i, :], decode_v[:, i, :], cp_m[:, i, :], cp_v[:, i, :]))
            mask_kl = mask_kl + torch.mean(kl_normal(f_z1[:, i, :], q_v[:, i, :], cp_m[:, i, :], cp_v[:, i, :]))

        total_kl = 0.3 * torch.mean(kl_base) + kl_dag

        # Label supervision loss via structural equations g_u(u)
        label_loss = self.mse_loss(g_u, label.float())
        mask_loss = mask_kl + label_loss

        nelbo = rec_loss + beta * (total_kl + mask_loss)
        return nelbo, total_kl, rec_loss, decoded_logits, z_given_dag

    def loss(self, x: torch.Tensor, label: torch.Tensor):
        """Step loss and dictionary logger for training loops."""
        nelbo, kl, rec, _, _ = self.negative_elbo_bound(x, label)
        summaries = {
            'train/loss': nelbo.detach(),
            'gen/elbo': -nelbo.detach(),
            'gen/kl_z': kl.detach(),
            'gen/rec': rec.detach()
        }
        return nelbo, summaries


In [6]:
# ============================================================================
# Part 6: Neural Structural Causal Modules, Encoders, & Decoders (Fixed)
# ============================================================================

def dag_right_linear(input_tensor: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor = None) -> torch.Tensor:
    """Computes X @ W^T + b with matrix or batched tensor support."""
    if input_tensor.dim() == 2 and bias is not None:
        return torch.addmm(bias, input_tensor, weight.t())
    output = torch.matmul(input_tensor, weight.t())
    return output + bias if bias is not None else output


def dag_left_linear(input_tensor: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor = None) -> torch.Tensor:
    """Computes W @ X + b with batched tensor support."""
    if input_tensor.dim() == 2 and bias is not None:
        return torch.addmm(bias, input_tensor, weight.t())
    output = torch.matmul(weight, input_tensor)
    return output + bias if bias is not None else output


class MaskLayer(nn.Module):
    """
    Non-linear Structural Equation Model (SEM) mapping layer f_i(z_i).
    Applies dedicated non-linear MLPs per causal concept dimension.
    """
    def __init__(self, z_dim: int = 16, concept: int = 4, z2_dim: int = 4):
        super().__init__()
        self.z_dim = z_dim
        self.concept = concept
        self.z2_dim = z2_dim

        # Per-concept structural equation sub-networks
        self.nets = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.z2_dim, 32),
                nn.ELU(inplace=True),
                nn.Linear(32, self.z2_dim)
            ) for _ in range(self.concept)
        ])

        self.global_net = nn.Sequential(
            nn.Linear(self.z_dim, 32),
            nn.ELU(inplace=True),
            nn.Linear(32, self.z_dim)
        )

    def masked(self, z: torch.Tensor) -> torch.Tensor:
        z_flat = z.view(-1, self.z_dim)
        return self.global_net(z_flat)

    def mix(self, z: torch.Tensor) -> torch.Tensor:
        """
        Applies individual non-linear structural transformations to each concept node.
        Supports (batch, concept, z2_dim), (batch, concept), or (batch, z_dim).
        """
        batch_size = z.size(0)
        # Reshape to (batch, concept, z2_dim)
        z_nodes = z.view(batch_size, self.concept, self.z2_dim)

        transformed = [self.nets[i](z_nodes[:, i, :]) for i in range(self.concept)]
        return torch.cat(transformed, dim=1)  # (batch, concept * z2_dim)


class Attention(nn.Module):
    """Bilinear Cross-Attention mechanism for latent state dependency modeling."""
    def __init__(self, in_features: int = 4):
        super().__init__()
        self.M = nn.Parameter(torch.randn(in_features, in_features) * 0.02)
        self.sigmoid = nn.Sigmoid()

    def attention(self, z: torch.Tensor, e: torch.Tensor):
        """
        Computes attention-weighted latents: Softmax(Sigmoid(z M e^T)) e
        z, e: (batch, concept, z2_dim)
        """
        a = torch.matmul(torch.matmul(z, self.M), e.permute(0, 2, 1))
        a = self.sigmoid(a)
        weights = torch.softmax(a, dim=-1)
        e_tilde = torch.matmul(weights, e)
        return e_tilde, weights


class DagLayer(nn.Module):
    """
    Causal DAG Adjacency and Structural Routing Layer.
    Maintains learnable weighted adjacency matrix A such that: Z = (I - A^T)^{-1} E
    """
    def __init__(self, num_nodes: int = 4, initial_dag: bool = True):
        super().__init__()
        self.num_nodes = num_nodes

        # Initialize adjacency matrix
        adj = torch.zeros(num_nodes, num_nodes)
        if initial_dag and num_nodes >= 4:
            # Ground-truth dependency initialization:
            # Node 0 (ball_r) -> Node 1 (h), Node 2 (x_true)
            # Node 1 (h)      -> Node 2 (x_true)
            # Node 3 (deep)   -> Node 2 (x_true)
            adj[0, 1] = 1.0
            adj[0, 2] = 1.0
            adj[1, 2] = 1.0
            adj[3, 2] = 1.0

        self.A = nn.Parameter(adj)
        self.register_buffer('I', torch.eye(num_nodes))

    def get_adj(self) -> torch.Tensor:
        """Enforces zero-diagonal on adjacency matrix to prevent self-loops."""
        return self.A * (1.0 - self.I)

    def mask_z(self, z: torch.Tensor) -> torch.Tensor:
        """Propagates latents via adjacency: A^T @ z"""
        A = self.get_adj()
        return torch.matmul(A.t(), z)

    def mask_u(self, u: torch.Tensor) -> torch.Tensor:
        """Propagates supervision labels via adjacency: A^T @ u"""
        A = self.get_adj()
        u_expanded = u.unsqueeze(-1)  # (batch, num_nodes, 1)
        res = torch.matmul(A.t(), u_expanded)
        return res.squeeze(-1)

    def calculate_dag(self, m: torch.Tensor, v: torch.Tensor):
        """
        Solves structural causal equations: Z = (I - A^T)^{-1} Z_encoded
        m, v: (batch, num_nodes, z2_dim)
        """
        A = self.get_adj()
        # Add epsilon damping to prevent singular matrix inversion instabilities
        matrix_to_invert = self.I - A.t() + 1e-6 * self.I
        inv_op = torch.linalg.inv(matrix_to_invert)

        # Batched matrix transformation along the causal node dimension
        m_dag = torch.einsum('ij,bjk->bik', inv_op, m)
        return m_dag, v

    def calculate_cov(self, m: torch.Tensor, v: torch.Tensor):
        A = self.get_adj()
        matrix_to_invert = self.I - A + 1e-6 * self.I
        inv_op = torch.linalg.inv(matrix_to_invert)

        m_cov = torch.einsum('ij,bjk->bik', inv_op, m)
        v_cov = torch.einsum('ij,bjk->bik', inv_op, v)
        v_cov = torch.einsum('bjk,ik->bji', v_cov, inv_op)
        return m_cov, v_cov


class ConvEncoder(nn.Module):
    """Convolutional Feature Encoder from 96x96 RGB images to latent distributions."""
    def __init__(self, in_channels: int = 3, z_dim: int = 16):
        super().__init__()
        self.z_dim = z_dim

        self.encoder_net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=4, stride=2, padding=1),   # 48x48
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),           # 24x24
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),          # 12x12
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 128, kernel_size=4, stride=2, padding=1),         # 6x6
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten()
        )

        self.fc_mean = nn.Linear(128 * 6 * 6, z_dim)
        self.fc_var = nn.Linear(128 * 6 * 6, z_dim)

    def encode(self, x: torch.Tensor):
        features = self.encoder_net(x)
        mu = self.fc_mean(features)
        var = F.softplus(self.fc_var(features)) + 1e-7
        return mu, var


class ConvDecoder(nn.Module):
    """Individual concept Transpose-Convolutional Decoder producing 96x96 RGB images."""
    def __init__(self, in_dim: int = 4, out_channels: int = 3):
        super().__init__()
        self.project = nn.Linear(in_dim, 128 * 6 * 6)

        self.deconv_net = nn.Sequential(
            nn.ConvTranspose2d(128, 128, kernel_size=4, stride=2, padding=1),  # 12x12
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   # 24x24
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),    # 48x48
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(32, out_channels, kernel_size=4, stride=2, padding=1) # 96x96
        )

    def decode(self, z_concept: torch.Tensor) -> torch.Tensor:
        batch_size = z_concept.size(0)
        h = self.project(z_concept).view(batch_size, 128, 6, 6)
        return self.deconv_net(h)


class ConvDec(nn.Module):
    """
    Modular Multi-Concept Image Decoder.
    Renders distinct causal factor representations into separate image spaces and aggregates them.
    """
    def __init__(self, z_dim: int = 16, concept: int = 4, z2_dim: int = 4, out_channels: int = 3):
        super().__init__()
        self.z_dim = z_dim
        self.concept = concept
        self.z2_dim = z2_dim

        self.decoders = nn.ModuleList([
            ConvDecoder(in_dim=self.z2_dim, out_channels=out_channels)
            for _ in range(self.concept)
        ])

    def decode_sep(self, z: torch.Tensor, label: torch.Tensor = None):
        """Decodes each causal node subspace and computes the average reconstruction logits."""
        batch_size = z.size(0)
        z_nodes = z.view(batch_size, self.concept, self.z2_dim)

        decoded_components = [
            self.decoders[i].decode(z_nodes[:, i, :])
            for i in range(self.concept)
        ]

        # Combine independent concept maps via uniform ensemble average
        logits = torch.stack(decoded_components, dim=0).mean(dim=0)
        return logits, decoded_components


class MLPDecoder(nn.Module):
    """Dense Multi-Layer Perceptron Decoder for causal reconstruction."""
    def __init__(self, z_dim: int = 16, concept: int = 4, z2_dim: int = 4, channels: int = 3, img_size: int = 96):
        super().__init__()
        self.z_dim = z_dim
        self.concept = concept
        self.z2_dim = z2_dim
        self.output_dim = channels * img_size * img_size

        self.decoders = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.z2_dim, 300),
                nn.ELU(inplace=True),
                nn.Linear(300, 1024),
                nn.ELU(inplace=True),
                nn.Linear(1024, self.output_dim)
            ) for _ in range(self.concept)
        ])

    def decode_sep(self, z: torch.Tensor, label: torch.Tensor = None):
        batch_size = z.size(0)
        z_nodes = z.view(batch_size, self.concept, self.z2_dim)

        components = [self.decoders[i](z_nodes[:, i, :]) for i in range(self.concept)]
        logits = torch.stack(components, dim=0).mean(dim=0)
        return logits, components


In [ ]:
# ============================================================================
# Final Part: Training Loop, Acyclicity Regularization, & Visualization (Fixed)
# ============================================================================

class FixedSeed:
    """
    Context manager that temporarily sets fixed seeds across NumPy, PyTorch CPU, and CUDA.
    Restores original states upon exit.
    """
    def __init__(self, seed: int = 0):
        self.seed = seed
        self.np_state = None
        self.torch_state = None
        self.cuda_state = None

    def __enter__(self):
        self.np_state = np.random.get_state()
        self.torch_state = torch.get_rng_state()
        if torch.cuda.is_available():
            self.cuda_state = torch.cuda.get_rng_state_all()

        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def __exit__(self, exc_type, exc_value, traceback):
        np.random.set_state(self.np_state)
        torch.set_rng_state(self.torch_state)
        if torch.cuda.is_available() and self.cuda_state is not None:
            torch.cuda.set_rng_state_all(self.cuda_state)


class DeterministicWarmup:
    """
    Linear deterministic warm-up schedule for KL divergence / beta-VAE (Annealing).
    """
    def __init__(self, n: int = 100, t_max: float = 1.0):
        self.t = 0.0
        self.t_max = t_max
        self.inc = t_max / float(max(n, 1))

    def __iter__(self):
        return self

    def __next__(self) -> float:
        self.t = min(self.t + self.inc, self.t_max)
        return self.t


def prepare_writer(
    model_name: str,
    overwrite_existing: bool = False,
    log_root: str = 'logs',
    checkpoint_root: str = 'checkpoints'
) -> SummaryWriter:
    """Initializes TensorBoard SummaryWriter with automatic directory management."""
    log_dir = os.path.join(log_root, model_name)
    save_dir = os.path.join(checkpoint_root, model_name)

    if overwrite_existing:
        if os.path.exists(log_dir):
            shutil.rmtree(log_dir)
        if os.path.exists(save_dir):
            shutil.rmtree(save_dir)

    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    return SummaryWriter(log_dir=log_dir)


def log_summaries(writer: SummaryWriter, summaries: dict, global_step: int):
    """Logs metrics dictionary to TensorBoard."""
    if writer is None:
        return
    for tag, val in summaries.items():
        if isinstance(val, torch.Tensor):
            val = val.detach().cpu().item() if val.numel() == 1 else val.mean().item()
        writer.add_scalar(tag, val, global_step=global_step)
    writer.flush()


def save_model_by_name(model: nn.Module, checkpoint_dir: str = 'checkpoints'):
    """Saves model state dictionary by resolving `model.name` dynamically."""
    model_name = getattr(model, 'name', 'model')
    save_dir = os.path.join(checkpoint_dir, model_name)
    os.makedirs(save_dir, exist_ok=True)

    file_path = os.path.join(save_dir, 'model.pt')
    torch.save(model.state_dict(), file_path)
    print(f'Model checkpoint successfully saved to: {file_path}')


# ----------------------------------------------------------------------------
# CLI & Execution Setup
# ----------------------------------------------------------------------------

parser = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
parser.add_argument('--epoch_max', type=int, default=101, help="Number of training epochs")
parser.add_argument('--iter_save', type=int, default=10, help="Save model every n epochs")
parser.add_argument('--batch_size', type=int, default=64, help="Batch size for training")
parser.add_argument('--lr', type=float, default=1e-3, help="Learning rate")
parser.add_argument('--toy', type=str, default="flow_mask", help="The toy dataset identifier")
parser.add_argument('--initial', action='store_true', default=True, help="Initialize the DAG adjacency matrix")
parser.add_argument('--data_dir', type=str, default='./causal_data/flow_noise', help="Dataset directory")

# Parse known args to allow execution in interactive environments (like Jupyter/Colab)
args, unknown = parser.parse_known_args()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_name = f"causalvae_toy={args.toy}"
print(f"Device: {device} | Model Name: {model_name}")
pprint(vars(args))

# Ensure visualization and dataset directories exist
os.makedirs('./figs_vae/', exist_ok=True)
if not os.path.exists(args.data_dir):
    print("Dataset not found locally. Generating synthetic causal water dataset...")
    generate_flow_dataset(output_dir=args.data_dir)

# ----------------------------------------------------------------------------
# Instantiate Submodules and CausalVAE Architecture
# ----------------------------------------------------------------------------

z1_dim = 4  # 4 causal concepts: [r, h, x_true, hole]
z2_dim = 4  # 4 dimensions per concept
z_dim = z1_dim * z2_dim

encoder = ConvEncoder(in_channels=3, z_dim=z_dim)
decoder = ConvDec(z_dim=z_dim, concept=z1_dim, z2_dim=z2_dim, out_channels=3)
dag_layer = DagLayer(num_nodes=z1_dim, initial_dag=args.initial)
attention_layer = Attention(in_features=z2_dim)
mask_z_layer = MaskLayer(z_dim=z_dim, concept=z1_dim, z2_dim=z2_dim)
mask_u_layer = MaskLayer(z_dim=z1_dim, concept=z1_dim, z2_dim=1)

lvae = CausalVAE(
    encoder=encoder,
    decoder=decoder,
    dag_layer=dag_layer,
    attention_layer=attention_layer,
    mask_z_layer=mask_z_layer,
    mask_u_layer=mask_u_layer,
    name=model_name,
    z_dim=z_dim,
    z1_dim=z1_dim,
    z2_dim=z2_dim
).to(device)

# ----------------------------------------------------------------------------
# Dataloaders, Optimizers, and Warmup
# ----------------------------------------------------------------------------

train_loader = get_batch_unin_dataset_withlabel(args.data_dir, batch_size=args.batch_size, dataset="train")
optimizer = torch.optim.Adam(lvae.parameters(), lr=args.lr, betas=(0.9, 0.999))
writer = prepare_writer(model_name)
warmup = DeterministicWarmup(n=args.epoch_max // 2, t_max=1.0)

losses = []
kls = []
recs = []
global_step = 0

print("\nStarting Training...")
for epoch in range(args.epoch_max):
    lvae.train()
    total_loss = 0.0
    total_rec = 0.0
    total_kl = 0.0
    last_u = None
    last_recon = None
    beta = next(warmup)

    for u, l in train_loader:
        global_step += 1
        optimizer.zero_grad()

        u = torch.clamp(u.to(device), 0.0, 1.0)
        l = l.to(device)

        # 1. Compute Base Variational NELBO with Beta Annealing
        nelbo, kl, rec, reconstructed_logits, _ = lvae.negative_elbo_bound(u, l, beta=beta)

        # 2. Augmented Lagrangian NOTEARS Acyclicity Constraint on off-diagonal elements
        dag_param = lvae.dag.get_adj()
        h_a = _h_A(dag_param, dag_param.size(0))
        acyclicity_penalty = 3.0 * h_a + 0.5 * (h_a ** 2)

        # 3. Total Loss computation, Gradient Clipping & Backprop
        loss = nelbo + acyclicity_penalty
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lvae.parameters(), max_norm=5.0)
        optimizer.step()

        # Metrics accumulation
        total_loss += loss.item()
        total_kl += kl.item()
        total_rec += rec.item()

        last_u = u
        last_recon = reconstructed_logits

    num_batches = len(train_loader)
    avg_loss = total_loss / num_batches
    avg_kl = total_kl / num_batches
    avg_rec = total_rec / num_batches

    losses.append(avg_loss)
    kls.append(avg_kl)
    recs.append(avg_rec)

    # TensorBoard Logging
    log_summaries(writer, {
        'Loss/Total': avg_loss,
        'Loss/KL': avg_kl,
        'Loss/Reconstruction': avg_rec,
        'DAG/h_A': h_a.item(),
        'Hyper/Beta': beta
    }, global_step=epoch)

    # Save visual reconstructions at end of epoch
    if last_u is not None and last_recon is not None:
        recon_probs = torch.sigmoid(last_recon)
        save_image(last_u[0], f'figs_vae/ground_truth_epoch_{epoch}.png', value_range=(0, 1))
        save_image(recon_probs[0], f'figs_vae/reconstruction_epoch_{epoch}.png', value_range=(0, 1))

    print(f"Epoch [{epoch:03d}/{args.epoch_max:03d}] | Loss: {avg_loss:.4f} | KL: {avg_kl:.4f} | Rec: {avg_rec:.4f} | h(A): {h_a.item():.6f} | Beta: {beta:.2f}")

    if epoch % args.iter_save == 0 or epoch == args.epoch_max - 1:
        save_model_by_name(lvae)

writer.close()

# ----------------------------------------------------------------------------
# Evaluation Plots
# ----------------------------------------------------------------------------

epochs_range = range(args.epoch_max)
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, losses, label='Total Loss (NELBO + DAG)', color='red')
plt.plot(epochs_range, kls, label='KL Divergence', color='green')
plt.plot(epochs_range, recs, label='Reconstruction Loss', color='blue')
plt.xlabel('Epochs')
plt.ylabel('Value')
plt.title('CausalVAE Training Convergence')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('figs_vae/training_convergence.png', dpi=150)
plt.close()
print("Training complete. Convergence plot saved to 'figs_vae/training_convergence.png'.")


Device: cpu | Model Name: causalvae_toy=flow_mask
{'batch_size': 64,
 'data_dir': './causal_data/flow_noise',
 'epoch_max': 101,
 'initial': True,
 'iter_save': 10,
 'lr': 0.001,
 'toy': 'flow_mask'}
Dataset not found locally. Generating synthetic causal water dataset...
Generating synthetic causal dataset at './causal_data/flow_noise'...
Rendered 1000/8100 images (13.7s elapsed)
Rendered 2000/8100 images (27.1s elapsed)
Rendered 3000/8100 images (40.6s elapsed)
Rendered 4000/8100 images (53.8s elapsed)
Rendered 5000/8100 images (67.2s elapsed)
Rendered 6000/8100 images (80.5s elapsed)
Rendered 7000/8100 images (94.0s elapsed)
Rendered 8000/8100 images (107.2s elapsed)
Finished generating 8100 causal samples in 108.51s.

Starting Training...
Epoch [000/101] | Loss: 2.9035 | KL: 8.9011 | Rec: 0.1862 | h(A): 0.013214 | Beta: 0.02
Model checkpoint successfully saved to: checkpoints/causalvae_toy=flow_mask/model.pt
Epoch [001/101] | Loss: 2.4215 | KL: 6.2341 | Rec: 0.0968 | h(A): 0.054989 